In [ ]:
import sys
from pathlib import Path

# Add src/ to Python path for local module imports
sys.path.insert(0, str(Path.cwd().parent / "src"))

# Questo notebook in realtà è da ignorare e non è mai stato utilizzato

In [1]:
from pathlib import Path


import os

from pprint import pprint

import constants

from mistralai import Mistral

from pathlib import Path

# Preliminari

In [2]:
# Configurazione Mistral
client = Mistral(
    api_key=os.getenv("MISTRAL_API_KEY")
)

# Configurazione WandB
#wandb.login(key=os.getenv("WANDB_API_KEY"))

# Parametri
base_dir = Path.cwd().parent
UPLOAD_DATA = True
#TEMPERATURE = 0.0
#ANN_MODEL = constants.RectalCancerStagingData
#SYSTEM_PROMPT_FILE_NAME = constants.SYSTEM_PROMPT_4

MODEL = constants.MISTRAL_LARGE_3

#Carica system prompt da file
#system_prompt_path = base_dir / "data" / "prompts" / SYSTEM_PROMPT_FILE_NAME
#system_prompt = create_system_prompt(system_prompt_path, ANN_MODEL)

# Upload data to Openai dashboard

## Create local fine-tuning datasets

Il dataset jsonl dovrà contenere righe con il formato chat completetion. Ogni riga dovrà essere:

```
{
  "messages": [
    { "role": "system", "content": <system_prompt>}
    { "role": "user", "content": <report_text> },
    { "role": "assistant", "content": <correct_annotation> },
  ]
}
```
Si possono usare i file nello stesso formato adatto a Openai.

In [3]:
if UPLOAD_DATA:
    training_file_path = base_dir / 'data' / 'ft-dataset' / constants.OPENAI_TRAIN_FILE_NAME
    validation_file_path = base_dir / 'data' / 'ft-dataset' / constants.OPENAI_VALIDATION_FILE_NAME
    
    training_data = client.files.upload(
        file={
            "file_name": "training_file.jsonl",
            "content": open(training_file_path, "rb"),
        }
    )
    
    validation_data = client.files.upload(
        file={
            "file_name": "validation_file.jsonl",
            "content": open(validation_file_path, "rb"),
        }
    )

In [4]:
print("Training file ID:", training_data.id)
print("Validation file ID:", validation_data.id)

Training file ID: f14129b4-7b76-4f23-9a9c-2166594eb01a
Validation file ID: fb848627-d812-4920-9ae4-66a824ee33b9


# Fine tuning

In [5]:
print(MODEL)

mistral-large-latest


In [6]:
# create a fine-tuning job
fine_tuning_job = client.fine_tuning.jobs.create(
    model=MODEL,
    training_files=[{"file_id": training_data.id, "weight": 1}],
    validation_files=[validation_data.id],
    hyperparameters={
        "epochs": 3,
    },
    auto_start=False,
    integrations=[
        {
            "project": "PRIN",
            "api_key": os.getenv("WANDB_API_KEY"),
        }
    ]
)

SDKError: API error occurred: Status 422. Body: {"detail": "Model not available for this type of fine-tuning (completion). Available model(s): open-mistral-nemo,pixtral-12b-latest"}

In [8]:
fine_tuning_job = client.fine_tuning.jobs.create(
  training_file=training_file_id,
  validation_file=validation_file_id,
  model=MODEL,
  seed=constants.SEED,
  suffix="report-extractor",
  integrations=[
    {
      "type": "wandb",
      "wandb": {"project": "PRIN"}
    }
  ]
)

job_id = fine_tuning_job.id

In [9]:
print("Job ID:", fine_tuning_job.id)
print("Status:", fine_tuning_job.status)

Job ID: ftjob-dO4vXegWPXbozguBnOd9Ddnm
Status: validating_files


In [10]:
response = client.fine_tuning.jobs.retrieve(job_id)

print("Job ID:", response.id)
print("Status:", response.status)
print("Trained Tokens:", response.trained_tokens)

Job ID: ftjob-dO4vXegWPXbozguBnOd9Ddnm
Status: validating_files
Trained Tokens: None


In [13]:
response = client.fine_tuning.jobs.list_events(job_id)

events = response.data
events.reverse()

for event in events:
    print(event.message)

Created fine-tuning job: ftjob-dO4vXegWPXbozguBnOd9Ddnm
Validating training file: file-9KY5fZxjEjbnrGdDQyYj5i and validation file: file-NLiLjQ528ZD4QJauzbFKoE
Files validated, moving job to queued state
Fine-tuning job started


In [12]:
response = client.fine_tuning.jobs.retrieve(job_id)
fine_tuned_model_id = response.fine_tuned_model

if fine_tuned_model_id is None:
    raise RuntimeError(
        "Fine-tuned model ID not found. Your job has likely not been completed yet."
    )

print("Fine-tuned model ID:", fine_tuned_model_id)

RuntimeError: Fine-tuned model ID not found. Your job has likely not been completed yet.